## <a name="0">Finetuning Large Language Model on Amazon SageMaker </a>

**This notebook uses Amazon Bedroc FAQ data to fine-tune an LLM.**

**Please work top to bottom of this notebook and don't skip sections as this could lead to error messages due to missing code.**


## <a name="step1">Step 1: Check GPU Memory</a>

To check the GPU memory, run the below command.

In [ ]:
!nvidia-smi


# CHECK GPU MEMORY AVAILABILITY
# Why: Ensures sufficient GPU memory exists for model training.
# If memory is occupied >50%, close other notebooks to avoid out-of-memory errors.

**If you see your CUDA memory is occupied by more than half (like the image below), then you need to shutdown other running notebooks.**

<p style="padding: 10px; border: 1px solid black;">
<img src="images/memory.png" alt="drawing" width="800"/> <br/>

## <a name="step2">Step 2: Import libraries</a>

Run the below code block sequentially to import the necessary libraries, including the Hugging Face Transformers library and the PyTorch library, which is a dependency for Transformers. If you haven't install the packages, uncomment the below line and install them.


In [ ]:
%%capture
!pip install -r requirements.txt


# INSTALL REQUIRED DEPENDENCIES
# Why: Install all necessary Python packages (PyTorch, Transformers, PEFT, etc.)
# needed for the LLM fine-tuning workflow from requirements.txt

In [ ]:
%%capture

import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

import os
# Suppress TensorFlow/Keras imports in Transformers — avoids Keras 3 conflict
# in SageMaker Distribution v4.4.1 which ships Keras 3 by default
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
# Reduce CUDA memory fragmentation — critical when GPU headroom is tight
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Guard for datasets config.py bug: version.parse(None) TypeError on broken TF metadata
# Fixed in datasets>=2.21.1; patching packaging to be safe if old version still present
import importlib.metadata as _meta
_orig_version = _meta.version
def _safe_version(pkg):
    v = _orig_version(pkg)
    return v if v is not None else '0.0.0'
_meta.version = _safe_version
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Tuple, Union
from datasets import Dataset, load_dataset, disable_caching
disable_caching() ## Disable huggingface cache to save memory

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from transformers import TextDataset

import torch
from torch.utils.data import Dataset, random_split
from transformers import TrainingArguments, Trainer
import accelerate
import bitsandbytes

from IPython.display import Markdown


# IMPORT ALL NECESSARY LIBRARIES FOR LLM FINE-TUNING
# Why: Load essential libraries for data processing, model loading, and training
# - warnings: Suppress non-critical warnings for cleaner output
# - pandas/numpy: Data manipulation and numerical operations
# - datasets: Hugging Face library for efficient dataset handling
# - transformers: Pre-trained models and tokenizers
# - torch: PyTorch deep learning framework
# - accelerate/bitsandbytes: Training optimization and 8-bit quantization

## <a name="step3">Step 3: Prepare the training dataset</a>

Load and view the dataset. You will use [Amazon Bedrock FAQs](https://aws.amazon.com/bedrock/faqs/) as the main dataset. The dataset has two columns `instruction` and `response`.

In [ ]:
bedrock_faqs_dataset = load_dataset("csv",
                                      data_files='data/amazon_bedrock_faqs.csv')['train']
bedrock_faqs_dataset


# LOAD TRAINING DATASET FROM CSV FILE
# Why: Load Amazon Bedrock FAQ data containing instruction-response pairs.
# Dataset structure: 'instruction' (questions) and 'response' (answers) columns
# Used for fine-tuning the LLM to answer domain-specific questions.

In [ ]:
bedrock_faqs_dataset[0]


# DISPLAY FIRST DATA SAMPLE
# Why: Verify the structure and content of a single dataset record
# to understand instruction-response format before preprocessing.

## <a name="step3">Step 3.1: Prepare the Prompt</a>
To finetune our LLM, you need to decorate the instruction dataset with a PROMPT like below.

In [ ]:
from utils.helpers import INTRO_BLURB, INSTRUCTION_KEY, RESPONSE_KEY, END_KEY, RESPONSE_KEY_NL, DEFAULT_SEED, PROMPT
'''
PROMPT = """{intro}
            {instruction_key}
            {instruction}
            {response_key}
            {response}
            {end_key}"""
'''
Markdown(PROMPT)


# IMPORT AND DISPLAY PROMPT TEMPLATE
# Why: Load the predefined prompt template structure from helpers module.
# The template formats instruction-response pairs into a standardized prompt format
# for the model to learn the correct response pattern during fine-tuning.

Now feed the PROMPT to the dataset via the below function named `_add_text`. It takes a record as input. The function first checks if both the instruction and response fields are not empty. If either of them is empty, it raises a ValueError with a corresponding error message. If both fields have values, the function creates a new "text" field in the record by formatting them using given PROMPT.

In [ ]:
def _add_text(rec):
    instruction = rec["instruction"]
    response = rec["response"]

    # Validate that required fields exist
    if not instruction:
        raise ValueError(f"Expected an instruction in: {rec}")

    if not response:
        raise ValueError(f"Expected a response in: {rec}")

    # Apply prompt template to combine instruction and response
    rec["text"] = PROMPT.format(
        instruction=instruction, response=response)

    return rec


# PREPARE DATASET RECORDS WITH FORMATTED PROMPT TEMPLATE
# Why: Transform raw instruction-response pairs into structured text format
# that the LLM will learn from. Validates non-null fields and applies prompt template
# to each record for consistent formatting during fine-tuning.

In [ ]:
bedrock_faqs_dataset = bedrock_faqs_dataset.map(_add_text)
bedrock_faqs_dataset[0]


# APPLY PROMPT TEMPLATE TO ENTIRE DATASET
# Why: Map the _add_text function across all dataset records to create
# a 'text' field with formatted prompts for each instruction-response pair.

Use `Markdown` to neatly display the text with PROMPT.

In [ ]:
Markdown(bedrock_faqs_dataset[0]['text'])


# DISPLAY FORMATTED PROMPT IN MARKDOWN
# Why: Render the first processed record's formatted text in readable markdown format
# to visually verify that the prompt template was applied correctly.

### <a name="#step4">Step 4: Load a pretrained LLM</a>


To load a pretrained model, initialize a tokenizer and a base model using the `databricks/dolly-v2-3b` model from the Hugging Face Transformers library. The tokenizer converts raw text into tokens, and the base model generates text based on a given prompt. By following the instructions outlined above, you can correctly instantiate these components and leverage their functionality in your code.


The `AutoTokenizer.from_pretrained()` function is used to instantiate the tokenizer.
- `padding_side="left"` specifies the side of the sequences where padding tokens will be added. In this case, padding tokens will be added to the left side of each sequence.
- The `eos_token` is a special token representing the end of a sequence. By assigning it to the `pad_token`, any padding tokens added during tokenization will also be considered as end-of-sequence tokens. This can be useful when generating text using the model, as it will know when to stop generating text after encountering padding tokens.
- The `tokenizer.add_special_tokens...` adds three additional special tokens to the tokenizer's vocabulary. These tokens likely serve specific purposes in the application using the tokenizer. For example, they could be used to mark the end of an input, an instruction, or a response in a dialogue system.

After execution, the `tokenizer` object will be initialized and ready to use for tokenizing text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("verseAI/databricks-dolly-v2-3b",
                                          padding_side="left")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({"additional_special_tokens":
                              [END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL]})


# INITIALIZE TOKENIZER FOR MODEL PREPROCESSING
# Why: Create a tokenizer that converts raw text into token IDs that the model understands.
# - padding_side="left": Pad sequences on the left to handle variable-length inputs
# - pad_token = eos_token: Treat padding as end-of-sequence to signal model when to stop
# - add_special_tokens: Register custom tokens for proper instruction-response structure

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "verseAI/databricks-dolly-v2-3b",
    torch_dtype=torch.float16,  # FP16 halves memory vs FP32 (~6GB instead of ~12GB)
    load_in_8bit=True,          # 8-bit quantization reduces to ~3-4GB — required for T4 16GB
    device_map="auto"
)


# LOAD PRE-TRAINED MODEL WITH 8-BIT QUANTIZATION
# Why: T4 GPU has 14.56 GiB total. FP32 Dolly-3B needs ~12GB just to load,
# leaving no room for gradients or activations during training.
# - torch_dtype=float16: Halves base memory footprint
# - load_in_8bit=True: bitsandbytes INT8 quantization brings model to ~3-4GB
# - device_map="auto": Distribute across available GPU(s) automatically
# Result: ~10GB free VRAM for activations, gradients, and LoRA adapters

### <a name="#step4.1">Step 4.1: Prepare model for training</a>
Some pre-processing needs to be done before training such an int8 model using peft, therefore import an utiliy function `prepare_model_for_int8_training` that will:

- Casts all the non `int8` modules to full precision (fp32) for stability
- Add a forward_hook to the input embedding layer to enable gradient computation of the input hidden states
- Enable gradient checkpointing for more memory-efficient training

In [ ]:
model.resize_token_embeddings(len(tokenizer))


# RESIZE MODEL TOKEN EMBEDDINGS
# Why: Expand the model's token embedding layer to accommodate new special tokens
# added to the tokenizer (END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL).
# This ensures the model can properly process these custom tokens during training.

Use the `preprocess_batch` function to preprocess the "text" field of the batch, applying tokenization, truncation, and other relevant operations based on the specified maximum length. It takes a batch of data, a tokenizer, and a maximum length as input.

Refer to `mlu_utils/helpers.py` file for more details.

In [ ]:
from functools import partial
from utils.helpers import mlu_preprocess_batch

MAX_LENGTH = 256  # Maximum token sequence length
_preprocessing_function = partial(mlu_preprocess_batch, max_length=MAX_LENGTH, tokenizer=tokenizer)


# DEFINE PREPROCESSING FUNCTION WITH TOKENIZATION
# Why: Create a partial function that tokenizes, truncates, and pads text to MAX_LENGTH.
# - MAX_LENGTH=256: Limits sequence length to fit model's context window efficiently
# - partial(): Pre-configures preprocessing function with tokenizer and max_length
# This function will be applied to all dataset batches before training.

Next, apply the preprocessing function to each batch in the dataset, modifying the "text" field accordingly. The map operation is performed in a batched manner and the "instruction", "response", and "text" columns are removed from the dataset. Finally, the `processed_dataset` is created by filtering the `bedrock_faqs_dataset` based on the length of the "input_ids" field, ensuring it is less than the specified `MAX_LENGTH`.

In [ ]:
encoded_bedrock_faqs_dataset = bedrock_faqs_dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "response", "text"],
)

processed_dataset = encoded_bedrock_faqs_dataset.filter(lambda rec: len(rec["input_ids"]) < MAX_LENGTH)


# TOKENIZE DATASET AND FILTER BY SEQUENCE LENGTH
# Why: Apply tokenization preprocessing to all dataset batches:
# - batched=True: Process multiple samples at once for efficiency
# - remove_columns: Drop original text columns to save memory
# - filter(): Remove sequences exceeding MAX_LENGTH to prevent training errors
# Result: A dataset with 'input_ids' and 'attention_mask' ready for model training

Split dataset into `train` and `test` for evaluation.

In [ ]:
split_dataset = processed_dataset.train_test_split(test_size=14, seed=0)
split_dataset


# SPLIT DATASET INTO TRAINING AND EVALUATION SETS
# Why: Create separate train (90%) and test (10%) splits for model evaluation:
# - test_size=14: Reserve 14 samples for validation testing
# - seed=0: Use fixed seed for reproducible splits across runs
# This allows monitoring model generalization on unseen data during training.

### <a name="#step5">Step 5: Define the trainer and finetuned the LLM</a>

To finetune a model efficiently, You will use [LoRA: Low-Rank Adaptation](https://arxiv.org/abs/2106.09685). LoRA, which freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times.


#### <a name="#step5.1">Step 5.1: Define the `LoraConfig` and load LoRA model</a>

You'll us build LoRA class `LoraConfig` from [huggingface 🤗 PEFT: State-of-the-art Parameter-Efficient Fine-Tuning](https://github.com/huggingface/peft). Within `LoraConfig`, let's specify the following parameters:

- `r`, the dimension of the low-rank matrices
- `lora_alpha`, the scaling factor for the low-rank matrices
- `lora_dropout`, the dropout probability of the LoRA layers

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
# from peft.utils import prepare_model_for_kbit_training  # Only needed for 8-bit quantization

# model = prepare_model_for_kbit_training(model)  # Commented out - using full precision
for param in model.parameters():
    if param.dtype == torch.float32 or param.dtype == torch.float16:
        param.requires_grad = False
num_layers_to_unfreeze = 2
for i, layer in enumerate(model.gpt_neox.layers[-num_layers_to_unfreeze:]):
    for param in layer.parameters():
        if param.dtype == torch.float32 or param.dtype == torch.float16:
            param.requires_grad = True
MICRO_BATCH_SIZE = 8
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE
LORA_R = 256
LORA_ALPHA = 512
LORA_DROPOUT = 0.05
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# CONFIGURE AND APPLY LORA ADAPTER FOR PARAMETER-EFFICIENT FINE-TUNING
# Why: Use LoRA (Low-Rank Adaptation) to reduce trainable parameters by 10,000x:
# 1. Prepare model for 8-bit training (enables mixed precision support)
# 2. Freeze all base model parameters (prevents catastrophic forgetting)
# 3. Unfreeze only top 2 layers (allows task-specific adaptation)
# 4. LoRA creates small adapter matrices (r=256, alpha=512) instead of fine-tuning entire model
#    This drastically reduces GPU memory and training time while maintaining performance.

Use the `get_peft_model` function to initialize the model with the LoRA framework, configuring it based on the provided `lora_config` settings. This allows the model to incorporate the benefits and capabilities of the LoRA optimization approach.

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# WRAP MODEL WITH LORA ADAPTER AND DISPLAY TRAINABLE PARAMETERS
# Why: Apply LoRA configuration to the model, reducing trainable parameters from millions to thousands.
# print_trainable_parameters() shows the efficiency gain (typically ~3% of original model size).
# This dramatic reduction enables training on consumer-grade GPUs without full fine-tuning overhead.

As you can see above, LoRA only trainable parameters is only ~3% of the full weights. Much efficient!

#### <a name="#step5.2">Step 5.2: Define the data collator</a>

A DataCollator is a huggingface🤗 transformers function that takes a list of samples from a Dataset and collate them into a batch, as a dictionary of PyTorch tensors.

Use `DataCollatorForCompletionOnlyLM`, which extends the functionality of the base `DataCollatorForLanguageModeling` class from transformers library. This custom collator is designed to handle examples where a prompt is followed by a response in the input text and modifies the labels accordingly.

Refer to `utils/helpers.py` for the implementation.

In [ ]:
from utils.helpers import MLUDataCollatorForCompletionOnlyLM

data_collator = MLUDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer, mlm=False, return_tensors="pt", pad_to_multiple_of=8
)


# INITIALIZE DATA COLLATOR FOR BATCH PREPROCESSING
# Why: Create a custom data collator that:
# - Handles instruction-response pairs intelligently
# - Only computes loss on response tokens (ignores instruction tokens)
# - Applies dynamic padding to variable-length sequences
# - Pads to multiple of 8 for hardware efficiency
# This ensures the model learns to generate correct responses, not just repeat instructions.

#### <a name="#step5.3">Step 5.3: Define the trainer</a>

To finetune the LLM, you need to define a trainer. First, define a training arguments.

In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-4
MODEL_SAVE_FOLDER_NAME = "dolly-3b-lora"

training_args = TrainingArguments(
                    output_dir=MODEL_SAVE_FOLDER_NAME,
                    fp16=True,   # Must match load_in_8bit=True — FP16 mixed precision training
                    per_device_train_batch_size=8,
                    per_device_eval_batch_size=8,
                    learning_rate=LEARNING_RATE,
                    num_train_epochs=EPOCHS,
                    logging_strategy="steps",
                    logging_steps=100,  # Log metrics every 100 steps
                    eval_strategy="steps",
                    eval_steps=100,  # Evaluate every 100 steps
                    save_strategy="steps",
                    save_steps=20000,  # Save checkpoint every 20000 steps
                    save_total_limit=10,  # Keep only 10 most recent checkpoints
                    optim="adamw_torch"
)


# CONFIGURE TRAINING HYPERPARAMETERS AND STRATEGY
# Why: Define the training process configuration:
# - EPOCHS=10: Train for 10 passes through the dataset for sufficient convergence
# - LEARNING_RATE=1e-4: Conservative learning rate for fine-tuning (lower than from-scratch training)
# - fp16=True: Use 16-bit floating point for 2x memory savings and faster computation
# - per_device_batch_size=1: Small batch size for memory-constrained environments (gradient accumulation compensates)
# - eval_steps=100: Evaluate on test set every 100 steps to monitor overfitting
# - save_steps=20000: Save model checkpoints at intervals for recovery
# - adamw_torch: Stable optimizer variant with weight decay regularization

Now is where the magic happen! Initialize the trainer with the defined model, tokenizer, training arguments, data collator and the train/eval datasets.

The training may take ~10 minutes. Once the training is done, you will save the finetuned model and tokenizer.

In [ ]:
trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=split_dataset['train'],
        eval_dataset=split_dataset["test"],
        data_collator=data_collator,
)
model.config.use_cache = False  # Disable cache during training to save memory

# Suppress bitsandbytes logging to show only progress bar
import logging
logging.getLogger('bitsandbytes').setLevel(logging.ERROR)
logging.getLogger('transformers.trainer').setLevel(logging.WARNING)

trainer.train()  # Start fine-tuning


# INITIALIZE TRAINER AND START FINE-TUNING PROCESS
# Why: Create a trainer object that orchestrates the entire training loop:
# - Combines model, tokenizer, training arguments, and datasets
# - Handles forward/backward passes, gradient updates, and validation
# - use_cache=False: Disable KV-cache during training (saves memory, enabled later for inference)
# - trainer.train(): Execute the fine-tuning process for all epochs
# Typically takes 10-15 minutes on GPU depending on dataset size.

#### <a name="#step5.4">Step 5.4: Save the finetuned model</a>


After the training finished, you can save the model to a directory using the [`transformers.PreTrainedModel.save_pretrained`] function.
This function only saves the incremental 🤗 PEFT weights (adapter_model.bin) that were trained, meaning it is super efficient to store, transfer, and load.

In [ ]:
trainer.model.save_pretrained(MODEL_SAVE_FOLDER_NAME)


# SAVE PEFT ADAPTER WEIGHTS TO DISK
# Why: Save only the LoRA adapter weights (adapter_model.bin) instead of full model.
# Advantages:
# - Adapter file is ~100MB vs ~7GB for full model (70x smaller)
# - Faster to download and transfer
# - Can be combined with base model for inference
# - Efficient for model deployment and sharing

If you want to save the full model you just finetuned, you can simply use the [`transformers.trainer.save_model`] function. Meanwhile, save the training arguments together with the trained model.

In [ ]:
trainer.save_model()


# SAVE TRAINER STATE AND TRAINING CONFIGURATION
# Why: Preserve complete training state including:
# - Training arguments and hyperparameters
# - Optimizer state
# - Training logs and metrics
# Useful for resuming training or analyzing training history.

In [ ]:
trainer.model.config.save_pretrained(MODEL_SAVE_FOLDER_NAME)


# SAVE MODEL CONFIGURATION
# Why: Persist the model's configuration settings including:
# - Architecture details (number of layers, hidden dimensions, etc.)
# - Attention mechanisms
# - Any customizations made during fine-tuning
# Essential for loading and using the model later without errors.

Let's save the tokenizer along with the trained model

In [ ]:
tokenizer.save_pretrained(MODEL_SAVE_FOLDER_NAME)


# SAVE TOKENIZER ALONG WITH TRAINED MODEL
# Why: Persist the fine-tuned tokenizer with the model:
# - Ensures consistent text-to-token encoding during inference
# - Custom special tokens (added during training) are preserved
# - Allows loading model and tokenizer together for deployment
# Critical for reproducible inference results.

### <a name="#step6">Step 6: Deploy the fine tuned model</a>

### <a name="step6">Overview of deployment parameters</a>

To deploy using the SageMaker Python SDK with the DJL, you will need to instantiate `Model` class with the following parameters:
```{python}
model = Model(
    image_uri,
    model_data=...,
    predictor_cls=...,
    role=aws_role
)
```
- `image_uri`: The Docker image URI representing the deep learning framework and version to be used.
- `model_data`: The location of the finetuned LLM model artifact in an S3 bucket. It specifies the path to the TAR GZ file containing the model's parameters, architecture, and any necessary artifacts.
- `predictor_cls`: This is just a "json in json out" predictor, nothing DJL related, check more details at [sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor).
- `role`: The AWS Identity and Access Management (IAM) role ARN that provides necessary permissions to access resources like the S3 bucket containing the model data.

### <a name="step6.1">Step 6.1: Instantiate SageMaker parameters</a>

Initialize a SageMaker session and retrieve information related to the AWS environment such as SageMaker role and AWS region. You also specify the image URI for a specific version of the "djl-deepspeed" framework using the SageMaker session's region. The image URI is a unique identifier for a specific Docker container image that can be used in various AWS services, such as Amazon SageMaker or Elastic Container Registry (ECR).

In [ ]:
%%capture
# Removed: sagemaker version managed by environment (v4.4.1 = 3.12.0)
# !pip3 install sagemaker==2.237.1


# INSTALL SAGEMAKER SDK FOR AWS DEPLOYMENT
# Why: Install the SageMaker Python SDK to enable programmatic deployment
# of the fine-tuned model to AWS SageMaker endpoints for production inference.
# Version 2.237.1 ensures compatibility with DJL deployment framework.

In [ ]:
import boto3
import json
from sagemaker.serve import ModelBuilder
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris

sagemaker_session = Session()
print("sagemaker_session: ", sagemaker_session)

aws_role = sagemaker_session.get_caller_identity_arn()
print("aws_role: ", aws_role)

aws_region = boto3.Session().region_name
print("aws_region: ", aws_region)

# Retrieve the DJL DeepSpeed inference container image for the specified region
image_uri = image_uris.retrieve(framework="djl-deepspeed",
                                version="0.26.0",
                                region=sagemaker_session._region_name)
print("image_uri: ", image_uri)


# INITIALIZE AWS AND SAGEMAKER PARAMETERS FOR DEPLOYMENT
# Why: Set up AWS configuration required for model deployment:
# SDK v3 import paths: sagemaker.core.helper.session_helper.Session,
# sagemaker.core.image_uris, sagemaker.serve.ModelBuilder


### <a name="step6.2">Step 6.2: Create the model artifact</a> ###

To upload the model artifact in the S3 bucket, we need to create TAR GZ file containing the model's parameters. First, you create a directory named `lora_model` and a subdirectory named `dolly-3b-lora`. The "-p" option ensures that the command creates any intermediate directories if they don't exist. Then, you copy the lora checkpoints `adapter_model.bin` and `adapter_config.json` to `dolly-3b-lora`. The base dolly model will be downloaded at runtime from huggingface hub.

In [ ]:
%%bash
set -e

# Verify training output exists before packaging
if [ ! -d "dolly-3b-lora" ]; then
  echo "ERROR: dolly-3b-lora/ not found. Run the training cell (Step 5) first."
  exit 1
fi

rm -rf lora_model
mkdir -p lora_model/dolly-3b-lora

cp dolly-3b-lora/adapter_config.json lora_model/dolly-3b-lora/

# PEFT>=0.4 saves safetensors by default; fall back to .bin for older versions
if [ -f "dolly-3b-lora/adapter_model.safetensors" ]; then
  cp dolly-3b-lora/adapter_model.safetensors lora_model/dolly-3b-lora/
elif [ -f "dolly-3b-lora/adapter_model.bin" ]; then
  cp dolly-3b-lora/adapter_model.bin lora_model/dolly-3b-lora/
else
  echo "ERROR: No adapter weights found in dolly-3b-lora/ (expected .safetensors or .bin)"
  ls dolly-3b-lora/
  exit 1
fi

echo "Model artifact ready:"
ls -lh lora_model/dolly-3b-lora/

# CREATE MODEL ARTIFACT DIRECTORY STRUCTURE FOR DEPLOYMENT
# Why: Prepare a deployment-ready package containing:
# - adapter_config.json & adapter weights: LoRA weights (loaded at runtime)
# - Base Dolly model: Downloaded from Hugging Face during deployment
# The directory structure matches DJL serving requirements for LoRA model serving.

Next, set the [DJL Serving configuration options](https://docs.aws.amazon.com/sagemaker/latest/dg/large-model-inference-configuration.html) in `serving.properties`. The jupyter `%%writefile` magic command enables us to write the following content to a file named "lora_model/serving.properties".
- `engine=Python`: This line specifies the engine used for serving.
- `option.entryPoint=model.py`: This line specifies the entry point for the serving process, which is set to "model.py".
- `option.adapter_checkpoint=/opt/ml/model/lora_model/dolly-3b-lora`: Full path to the LoRA adapter checkpoint in the container.
- `option.adapter_name=dolly-lora`: This line sets the name of the adapter to "dolly-lora", a component that helps interface between the model and the serving infrastructure.

In [ ]:
%%writefile lora_model/serving.properties
engine=Python
option.entryPoint=model.py
option.adapter_checkpoint=/opt/ml/model/lora_model/dolly-3b-lora
option.adapter_name=dolly-lora
# Force single CUDA runtime — prevents bitsandbytes multi-libcudart warning in DJL 0.26.0
option.env.BITSANDBYTES_NOWELCOME=1
option.env.BNB_CUDA_VERSION=121
option.env.TRANSFORMERS_NO_BITSANDBYTES=1


# CONFIGURE DJL SERVING PROPERTIES FOR LORA MODEL DEPLOYMENT
# Why: Define how DJL serving framework loads and serves the fine-tuned model:
# - engine=Python: Use Python runtime for DJL service
# - entryPoint=model.py: Custom Python script handling model loading and inference
# - adapter_checkpoint=dolly-3b-lora: Path to LoRA adapter weights
# - adapter_name=dolly-lora: Identifier for the LoRA adapter for the serving framework

Another file you need in the the model artifact is the environment requirement file. Create a file named `lora_model/requirements.txt` and write a list of Python package requirements, typically used with package managers like `pip`.

In [ ]:
%%writefile lora_model/requirements.txt
tokenizers>=0.19,<0.20
peft==0.14.0

### <a name="step6.3">Step 6.3: Create the inference script</a>

Similar to the finetuning notebook, a custom pipeline `InstructionTextGenerationPipeline` is defined. The code is provided in `utils/deployment_model.py`.

You save these inference functions to `lora_model/model.py`.

In [ ]:
%%bash
cp utils/deployment_model.py lora_model/model.py


# COPY CUSTOM INFERENCE HANDLER TO MODEL ARTIFACT
# Why: Deploy the custom model.py inference script (from utils/deployment_model.py) which:
# - Loads the base Dolly model and LoRA adapter at inference time
# - Handles request preprocessing and tokenization
# - Performs model inference with loaded fine-tuned weights
# - Formats responses for end users

### <a name="step6.4">Step 6.4: Upload the model artifact to S3</a>

Create a compressed tarball archive of the "lora_model" directory and saves it as "lora_model.tar.gz".

In [ ]:
%%bash
tar -cvzf dolly_lora_model.tar.gz lora_model/


# CREATE COMPRESSED TARBALL OF MODEL ARTIFACT DIRECTORY
# Why: Package lora_model/ directory and all contents into a single compressed file
# for efficient S3 upload and SageMaker deployment:
# - Reduces upload size significantly
# - Required format for SageMaker model artifact versioning
# - -cvzf: Create (c), verbose (v), compressed gzip (z), from file (f)

Upload the "lora_model.tar.gz" file to the specified S3 bucket.

In [ ]:
import boto3
import json
from sagemaker.serve import ModelBuilder
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris

s3 = boto3.resource('s3')
s3_client = boto3.client('s3')

s3 = boto3.resource('s3')

# Locate the artifact bucket (AWS Lab environment has specific naming convention)
for bucket in s3.buckets.all():
    if bucket.name.startswith('artifact'):  # Find bucket with 'artifact' prefix
        mybucket = bucket.name
        print(mybucket)

# Upload compressed model artifact to S3
response = s3_client.upload_file("dolly_lora_model.tar.gz", mybucket, "dolly_lora_model.tar.gz")


# UPLOAD COMPRESSED MODEL ARTIFACT TO AWS S3 BUCKET
# Why: Store the model tarball in S3 (SageMaker's model repository):
# - S3 is required for SageMaker to access model data during deployment
# - boto3 S3 client enables programmatic file uploads
# - Model must be accessible by ARN (s3://bucket-name/key-path)
# This URL will be provided to SageMaker during endpoint creation.

### <a name="step6.5">Step 6.5: Deploy the Model</a> ###

Now it's the time to deploy the finetuned LLM using SageMaker Python SDK. The SageMaker Python SDK `Model` class is instantiated with the following parameters:

- `image_uri`: The Docker image URI representing the deep learning framework and version to be used.
- `model_data`: The location of the finetuned LLM model artifact in an S3 bucket. It specifies the path to the TAR GZ file containing the model's parameters, architecture, and any necessary artifacts.
- `predictor_cls`: This is just a "json in json out" predictor, nothing DJL related, check more details at [sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor).
- `role`: The AWS Identity and Access Management (IAM) role ARN that provides necessary permissions to access resources like the S3 bucket containing the model data.
- `name`: The model name. If None, a default model name will be selected on each deploy.

In [ ]:
from datetime import datetime, timezone
timestamp_prefix = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S-%f")[:23]  # includes milliseconds

model_data="s3://{}/dolly_lora_model.tar.gz".format(mybucket)
model_name=f"dolly-model-{timestamp_prefix}"

# Clean up any leftover endpoint config with the same name before deploying
# Prevents: ValidationException: Cannot create already existing endpoint configuration
import botocore
sm_client = boto3.client('sagemaker')
endpoint_name = f"dolly-endpoint-{timestamp_prefix}"
try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"Cleaned up existing endpoint config: {endpoint_name}")
except sm_client.exceptions.ClientError:
    pass  # Config doesn't exist — nothing to clean up

import os
import getpass

# Enter your Hugging Face token — input is hidden like a password field
# Get a read token from: https://huggingface.co/settings/tokens
hf_token = os.environ.get("HUGGING_FACE_HUB_TOKEN") or getpass.getpass("Enter Hugging Face token: ")
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print("Token set ✓")

In [ ]:
model_builder = ModelBuilder(
    image_uri=image_uri,
    s3_model_data_url=model_data,
    role_arn=aws_role,
    instance_type="ml.g5.xlarge",
    env_vars={"HUGGING_FACE_HUB_TOKEN": hf_token}
)
model_builder.build()


# CREATE AND BUILD SAGEMAKER MODELBUILDER (SDK v3)
# Why: sagemaker.model.Model was removed in SDK v3. ModelBuilder is the replacement.
# - instance_type: MUST be set at constructor, not just deploy() — SDK v3 uses it
#   during build() to select the right container/config; omitting it defaults to ml.m5.large
# - env_vars: injects HUGGING_FACE_HUB_TOKEN into the container so deployment_model.py
#   can authenticate with HF Hub to download verseAI/databricks-dolly-v2-3b (private repo)
# build() registers the Model resource; deploy() creates the endpoint
# This Model object is then deployed to create an inference endpoint.

Note: **The deployment should finish within 10 minutes. If it took longer than that, your endpoint may be failed.**

In [ ]:
endpoint_name = f"dolly-endpoint-{timestamp_prefix}"

endpoint = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=endpoint_name,
    wait=False
)


# DEPLOY MODEL TO SAGEMAKER ENDPOINT WITH ASYNCHRONOUS INITIALIZATION
# Why: Create a production-ready inference endpoint:
# - 1 instance: Deploy one ml.g5.xlarge GPU instance (NVIDIA A10G GPU)
# - ml.g5.xlarge: GPU instance with 1x A10G GPU (24GB VRAM)
# - endpoint_name: Unique endpoint identifier with timestamp for tracking
# - wait=False: Return immediately; deployment continues in background (takes 5-10 minutes)
# After deployment, endpoint accepts inference requests via predictor object.

In [ ]:
import boto3
import time
sm_client = boto3.client('sagemaker')
while True:
    status = sm_client.describe_endpoint(EndpointName=endpoint.endpoint_name)['EndpointStatus']
    print(f"Status: {status}")
    if status == 'InService':  # Endpoint is ready for inference
        break
    time.sleep(30)  # Wait 30 seconds before checking again


# POLL ENDPOINT STATUS UNTIL DEPLOYMENT COMPLETES
# Why: Monitor deployment progress and wait for endpoint to become operational:
# - Checks endpoint status every 30 seconds
# - 'InService' status means endpoint is ready to accept inference requests
# - Deployment typically takes 5-10 minutes (longer on first deployment)
# Prevents sending requests before endpoint is ready (would cause failures).

### <a name="step7">Step 7: Test the deployed Inference</a>

Test the inference endpoint with [predictor.predict](https://sagemaker.readthedocs.io/en/stable/api/inference/predictors.html#sagemaker.predictor.Predictor.predict)!

In [ ]:
import json
payload = json.dumps({"inputs": "Is the content processed by Amazon Bedrock moved outside the AWS Region where I am using Amazon Bedrock?"}).encode('utf-8')
response = endpoint.invoke(body=payload)

# Parse the response body (SDK v3 returns a pydantic InvokeEndpointOutput object)
# response.body is a botocore StreamingBody -> attribute access, NOT response['body']
outputs = json.loads(response.body.read().decode('utf-8'))

# Print immediately so we see output even if display fails
print("="*50)
print("MODEL RESPONSE:")
print("="*50)
print(outputs)
print("="*50)


# SEND INFERENCE REQUEST TO DEPLOYED ENDPOINT
# Why: Test the fine-tuned model with a real question about Amazon Bedrock:
# - endpoint.invoke(body=...): SDK v3 replacement for predictor.predict()
# - response.body.read(): StreamingBody is exposed as an attribute on the pydantic
#   InvokeEndpointOutput model, so subscripting (response['body']) raises TypeError
# - json.loads(): Parse the JSON payload returned by model.py's inference handler
# - StreamingBody can only be read ONCE; re-running this cell needs a fresh invoke()
# This demonstrates the model's ability to answer domain-specific queries after fine-tuning.

In [ ]:
from IPython.display import Markdown, display

# Debug: check what we got
print(f"Type of outputs: {type(outputs)}")
print(f"Outputs value: {outputs}")
print()

# Normalize the response shape:
# - DJL pipeline can return: dict, str, or list
# - Dict format: {"instruction_text": "...", "generated_text": "..."}
# - String format: plain text response
# - List format: [{"generated_text": "..."}]

if isinstance(outputs, str):
    # Plain string response
    generated_text = outputs
elif isinstance(outputs, dict):
    # Dict with generated_text or instruction_text+generated_text
    generated_text = outputs.get("generated_text", str(outputs))
elif isinstance(outputs, list) and outputs:
    # List format
    first = outputs[0]
    if isinstance(first, dict):
        generated_text = first.get("generated_text", str(first))
    else:
        generated_text = str(first)
else:
    # Fallback
    generated_text = str(outputs)

print(f"Extracted text length: {len(generated_text)} characters")
print()

display(Markdown(f"**Response:**\n\n{generated_text}"))


# DISPLAY INFERENCE RESPONSE IN FORMATTED MARKDOWN
# Why: Render the model's response in readable markdown format for clean visualization.
# Shows the fine-tuned model's ability to generate coherent, relevant answers
# about the domain it was fine-tuned on (Amazon Bedrock FAQs).